# Sprint3_Part1_ML_EDA (1)

**목표**: EDA 기본적인 과정 경험  
**내용**: 학습 데이터 특징 이해, 모델 성능 향상방안 모색

-----------
# EDA & 데이터 전처리


## 데이터 불러오기 및 정보 확인

In [ ]:
# 학습 데이터, 평가 데이터 불러오기
# **test 셋은 타겟 칼럼 ['RentedBikeCount']이 없기 때문에 이후 추가적으로 같이 처리 해준다.
import pandas as pd

url_1 = 'https://raw.githubusercontent.com/rngus4656/datasets/refs/heads/main/S2/SeoulBikeData_train.csv'
url_2 = 'https://raw.githubusercontent.com/rngus4656/datasets/refs/heads/main/S2/SeoulBikeData_test.csv'
df_train = pd.read_csv(url_1)
df_test = pd.read_csv(url_2)

df_train.head()

In [ ]:
# test셋은 타겟칼럼이 미포함 된 것 확인(shape)
print(df_train.shape, df_test.shape)
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
df_train.describe(exclude='number')

### 결측치 & 중복 확인

In [ ]:
print('결측치\n', df_train.isna().sum())
print('중복값\n', df_train.duplicated().sum())

### 타겟 분포 확인  
> 타겟은 주로 정규분포를 이루도록 하지만, 데이터 특징에따라 방식을 정한다.

In [ ]:
# 타겟칼럼의 정규분포화가 필요할때는 로그변환, Box-Cox, Yeo-Johnson 등의 방법이 있다.
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df_train['Rented Bike Count'], kde=True, label=f'Skewness : {df_train['Rented Bike Count'].skew():.2f}')
plt.legend()
plt.show()

# Feature Engineering
> 학습 데이터를 위한 Feature Engineering 단계

In [ ]:
# 원본 수정 방지를 위한 copy 생성
df_train2 = df_train.copy()
df_test2 = df_test.copy()

In [ ]:
# S2P2 colab(3) 데이터 시각화에서 했던 feature_engineering 적용하는 함수 지정

def feature_engineering(df):
  df_fe = df.copy()
  # Date 월, 요일, 주말여부
  df_fe['Date'] = pd.to_datetime(df_fe['Date'], format='%d/%m/%Y')
  df_fe['month'] = df_fe['Date'].dt.month
  df_fe['dayname'] = df_fe['Date'].dt.day_name()
  df_fe['weekend'] = (df_fe['Date'].dt.weekday >= 5).astype(int)

  # 칼럼 단위 변환
  df_fe['Snowfall (cm)'] = df_fe['Snowfall (cm)'] / 10
  df_fe.rename(columns={'Snowfall (cm)': 'Snowfall (mm)'}, inplace=True)

  # WEP (mm) 칼럼 추가
  df_fe['WEP (mm)'] = df_fe['Rainfall(mm)'] + df_fe['Snowfall (mm)']

  # 시간대 카테고리 생성
  bins = [-1, 5, 11, 17, 23]
  labels = ['Dawn(0~5)', 'Morning(6~11)', 'Afternoon(12~17)', 'Evening(18~23)']
  df_fe['Hour_cat'] = pd.cut(df_fe['Hour'], bins=bins, labels=labels)


  # Date, id, Functioning Day 칼럼 제거 - 데이터 누수 방지, 불필요칼럼 제거
  df_fe = df_fe.drop(['Date', 'id', 'Functioning Day'], axis=1)

  return df_fe

In [ ]:
# df_train, df_test에 feature_engineering 함수 적용
fe_train = feature_engineering(df_train2)
fe_test = feature_engineering(df_test2)
fe_train.head()

# 학습 데이터셋 분리  
> 모델이 학습할 데이터 train  
학습모델을 검증할 데이터 val  
최종 성능을 평가할 데이터 test

In [ ]:
# train(문제집 학습), val(모의고사), test(수능)

from sklearn.model_selection import train_test_split

X = fe_train.drop('Rented Bike Count', axis=1) # 독립변수 X (학습 칼럼)
y = fe_train['Rented Bike Count'] # 종속변수 y (타겟 칼럼 분리)
X_test = fe_test.copy() # (예측해야 할 테스트 데이터 셋)

# 학습 데이터셋, 검증 데이터셋 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
X_train.shape, X_val.shape, X_test.shape

In [ ]:
X_train.head()

# 데이터 전처리  

> 데이터 전처리(Data Preprocessing)  
모델이 학습할 수 있도록 원본 데이터를 적절한 형태로 변환하는 과정이다.  
범주형 변수 인코딩, 스케일링 등의 작업을 포함한다.  
모델의 성능과 안정성에 직접적인 영향을 미치는 중요한 단계

In [ ]:
cat_cols = X.select_dtypes(exclude='number').columns
numeric = X.select_dtypes(include='number').columns
print('범주형 칼럼:', cat_cols)
print('수치형 칼럼:', numeric)

## 범주형 인코딩

In [ ]:
# One Hot Encoding - 가장 대표적인 범주형 인코딩 방식, 라벨 인코딩, 타겟 인코딩 등의 다양한 방식이 있다.

from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_ohe = ohe.fit_transform(X_train[cat_cols])
X_val_ohe = ohe.transform(X_val[cat_cols])
X_test_ohe = ohe.transform(X_test[cat_cols])

In [ ]:
# 인코딩된 값을 적용하기위해 데이터프레임 형식으로 변환
ohe_cat_cols = ohe.get_feature_names_out(cat_cols)

X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe_cat_cols)
X_val_ohe = pd.DataFrame(X_val_ohe, columns=ohe_cat_cols)
X_test_ohe = pd.DataFrame(X_test_ohe, columns=ohe_cat_cols)
X_train_ohe # X_train 정보와 비교해보면 쉽게 이해할 수 있다.

## 수치형 스케일링

In [ ]:
# 수치형 칼럼의 스케일링

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numeric])
X_val_scaled = scaler.transform(X_val[numeric])
X_test_scaled = scaler.transform(X_test[numeric])

X_train_scaled = pd.DataFrame(X_train_scaled, columns=numeric)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=numeric)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=numeric)
X_train_scaled.head()

## 인코딩 + 스케일링 데이터 합치기

In [ ]:
# 범주형, 수치형 전처리 데이터 concat

X_tr_EDA = pd.concat([X_train_ohe, X_train_scaled], axis=1)
X_va_EDA = pd.concat([X_val_ohe, X_val_scaled], axis=1)
X_te_EDA = pd.concat([X_test_ohe, X_test_scaled], axis=1)
X_tr_EDA.head() #훈련용 데이터셋 전처리 완료

# 데이터 전처리 자동화

> def 지정함수를 사용하면 쉽게 전처리를 자동화 할 수 있다.

In [ ]:
# 데이터셋 전처리 함수

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

def preprocessing(train_df, test_df, test_size=0.25, random_state=42, drop_cols=True):
  # 1) df복사 및 불필요 칼럼 삭제
  train2 = train_df.copy()
  test2  = test_df.copy()
  if drop_cols:
    train2 = train2.drop(['Date', 'id', 'Functioning Day'], axis=1)
    test2  = test2.drop(['Date', 'id', 'Functioning Day'], axis=1)


  # train_test_split
  X = train2.drop('Rented Bike Count', axis=1) # 독립변수 X (학습 칼럼)
  y = train2['Rented Bike Count'] # 종속변수 y (타겟 칼럼 분리)
  X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=test_size, random_state=random_state)

  # 범주형 칼럼 OneHotEncoding
  cat_cols = X_tr.select_dtypes(include=['object', 'category']).columns
  ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
  X_tr_ohe = ohe.fit_transform(X_tr[cat_cols])
  X_va_ohe = ohe.transform(X_va[cat_cols])
  X_te_ohe = ohe.transform(test2[cat_cols])
  ohe_cat_cols = ohe.get_feature_names_out(cat_cols)

  X_tr_ohe = pd.DataFrame(X_tr_ohe, columns=ohe_cat_cols)
  X_va_ohe = pd.DataFrame(X_va_ohe, columns=ohe_cat_cols)
  X_te_ohe = pd.DataFrame(X_te_ohe, columns=ohe_cat_cols)

  # 수치형 칼럼의 스케일링
  numeric = X_tr.select_dtypes(include=['int32', 'int64', 'float64']).columns

  scaler = StandardScaler()
  X_tr_scaled = scaler.fit_transform(X_tr[numeric])
  X_va_scaled = scaler.transform(X_va[numeric])
  X_te_scaled = scaler.transform(test2[numeric])

  X_tr_scaled = pd.DataFrame(X_tr_scaled, columns=numeric)
  X_va_scaled = pd.DataFrame(X_va_scaled, columns=numeric)
  X_te_scaled = pd.DataFrame(X_te_scaled, columns=numeric)

  # 범주형, 수치형 전처리 데이터 concat
  X_tr_2 = pd.concat([X_tr_ohe, X_tr_scaled], axis=1)
  X_va_2 = pd.concat([X_va_ohe, X_va_scaled], axis=1)
  X_te_2 = pd.concat([X_te_ohe, X_te_scaled], axis=1)
  print('훈련용 데이터셋 전처리 완료')

  return X_tr_2, X_va_2, X_te_2

In [ ]:
# EDA 미적용 baseline dataset 생성
X_tr_ba, X_va_ba, X_te_ba = preprocessing(df_train, df_test)

# EDA 적용 dataset 생성예시 (이미 위에서 만들었지만 다시 만들어야할 경우)
fe_train = feature_engineering(df_train)
fe_test = feature_engineering(df_test)
X_tr_EDA, X_va_EDA, X_te_EDA = preprocessing(fe_train, fe_test, drop_cols=False)

# 모델 학습 및 테스트

In [ ]:
# DT, RF, XGB 회귀모델 성능평가

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 의사결정나무, 랜덤 포레스트, XGB부스트 모델
dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
xgb = XGBRegressor(n_estimators=100, n_jobs=-1, random_state=42)


In [ ]:
# EDA 데이터셋 학습 및 성능평가 비교

X_tr = [X_tr_ba, X_tr_EDA]
X_va = [X_va_ba, X_va_EDA]
names = ['basic', 'EDA']

for i in range(len(names)):
  print(f"{names[i]}")
  # Decision Tree
  dt.fit(X_tr[i], y_train)
  dt_pred = dt.predict(X_va[i])

  # Random Forest
  rf.fit(X_tr[i], y_train)
  rf_pred = rf.predict(X_va[i])

  # XGBoost
  xgb.fit(X_tr[i], y_train)
  xgb_pred = xgb.predict(X_va[i])

  def evaluate(y_true, y_pred, model_name):
      rmse = np.sqrt(mean_squared_error(y_true, y_pred))
      r2 = r2_score(y_true, y_pred)
      print(f"{model_name}")
      print(f"RMSE: {rmse:.3f}")
      print(f"R2: {r2:.3f}")
      print("-"*30)

  evaluate(y_val, dt_pred, "Decision Tree")
  evaluate(y_val, rf_pred, "Random Forest")
  evaluate(y_val, xgb_pred, "XGBoost")

## Feature Importance 확인하기  

> Feature Importance
- 트리 기반 모델에서 불순도 감소량을 기준으로 계산되는 변수 중요도
- 여러 트리의 중요도를 평균하여 산출
- 연속형 변수나 범주 수가 많은 변수에 편향될 수 있음

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_feature_importance_subplot(models, X, model_names):

    fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 6))

    if len(models) == 1:
        axes = [axes]

    for ax, model, name in zip(axes, models, model_names):

        fi = pd.Series(model.feature_importances_,
                       index=X.columns).sort_values(ascending=False).head(20)

        fi.plot(kind="barh", ax=ax)

        ax.set_title(f"{name} Feature Importance")
        ax.set_xlabel("Importance")

        ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

In [ ]:
models = [dt, rf, xgb]
names = ["Decision Tree", "Random Forest", "XGBoost"]

plot_feature_importance_subplot(models, X_tr_EDA, names)

## Permutation Importance 확인하기

> Permutation Importance
- 특정 변수를 무작위로 섞었을 때 모델 성능 감소량으로 중요도 측정
- 모델 재학습 없이 계산 가능, Feature Importance보다 신뢰도가 높은편
- 변수 간 상관관계가 높으면 왜곡될 수 있음

In [ ]:
from sklearn.inspection import permutation_importance

def plot_permutation_importance_subplot(models, X_val, y_val, model_names):

    fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 6))

    if len(models) == 1:
        axes = [axes]

    for ax, model, name in zip(axes, models, model_names):

        result = permutation_importance(
            model,
            X_val,
            y_val,
            n_repeats=10,
            random_state=42,
            n_jobs=-1
        )

        perm_importance = pd.Series(
            result.importances_mean,
            index=X_val.columns
        ).sort_values(ascending=False).head(20)

        perm_importance.plot(kind="barh", ax=ax)

        ax.set_title(f"{name} Permutation Importance")
        ax.set_xlabel("Mean Decrease in Score")

        ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

In [ ]:
models = [dt, rf, xgb]
names = ["Decision Tree", "Random Forest", "XGBoost"]

plot_permutation_importance_subplot(models, X_va_EDA, y_val, names)

# 실습 목표
DAY 1) EDA 데이터셋에서 최소 한가지 이상의 피쳐 엔지니어링을 해보세요.

-----
# Sprint3_Part1_ML_EDA (2)

**목표**: ML 하이퍼파라미터 튜닝 및 성능 테스트  
**내용**: 머신러닝 하이퍼파라미터 조정, 성능평가, PCA/클러스터링 이해

# 모델 하이퍼 파라미터 튜닝 및 테스트  
따로 스크롤 올려가면서 확인하기 불편하신 분들은  
[하이퍼 파라미터 설명 페이지](https://github.com/rngus4656/wassup13/blob/main/Sprint3/DT_RF_XGB_%ED%95%98%EC%9D%B4%ED%8D%BC%ED%8C%8C%EB%9D%BC%EB%AF%B8%ED%84%B0.md)를 따로 띄워놓으셔도 됩니다.

## 의사결정나무 회귀모델(Decision TreeRegressor) 하이퍼 파라미터  
> 의사결정나무  
데이터를 분할 기준에 따라 반복적으로 나누어 예측값을 도출하는 기본적인 트리모델이다.  
각 분기 과정이 명확하게 드러나기 때문에 예측 근거를 설명하기 용이하다.

- 단순 트리구조로 성능이 불안정 할 수 있음
- 예측 근거가 중요한 도메인(금융, 의료 등)에 적합

참고 링크: [sklearn.tree
DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)

**1) 분할/손실 기준**
| 파라미터   | 의미 | 튜닝 포인트  |
| ------- | ----- | ---- |
| `criterion` | 노드 분할 품질을 평가하는 기준(회귀 손실) | 디폴트=`"squared_error"`(MSE) 고정 추천 |
| `splitter`  | 각 노드에서 분할을 고르는 방식  | 디폴트=`"best"` / `"random"`(랜덤 선택)   |


**2) 모델 복잡도(과적합 제어)**
| 파라미터 | 의미 | 튜닝 포인트 |
| --- | --- | --- |
| `max_depth` | 트리 최대 깊이 | 깊을수록 과적합↑ 주로 3~30 (1순위 제어 변수) |
| `min_samples_split` | 분할 가능한 최소 샘플 수 | 디플트=2 ↑하면 분할이 줄어 단순화 |
| `min_samples_leaf`  | 리프(말단) 노드 최소 샘플 수 | 디폴트=1 ↑하면 예측이 부드러워지고 과적합↓    |
| `max_leaf_nodes` | 최대 리프 노드 수 | 선택, 주로 10~1000 트리 크기 상한(안정적인 규제) |


**3) 추가 규제**
| 파라미터 | 의미 | 튜닝 포인트 |
| --- | --- | --- |
| `max_features` | 분할 시 고려할 feature 비율   | 선택, `"sqrt"`,"log2",`0~1` |
| `min_impurity_decrease` | 일정 값 이상 불순도 감소가 있어야 분할  | 디폴트=0 (0~0.05) ↑하면 보수적으로 분할(과적합↓) |
| `random_state` | 랜덤 시드 | 랜덤값 고정  |


## RandomForestRegressor 하이퍼파라미터  
> 랜덤 포레스트  
여러 개의 의사결정나무를 무작위 샘플링한 뒤,  
각 트리의 예측값을 평균하여 최종 예측을 수행하는 앙상블 모델이다.  
- 단일트리 대비 비교적 안정적이며 무난하고 좋은 성능을 보임
- 여러 트리를 평균하므로 예측값 근거 해석 어려움 (블랙박스)

참고 링크: [sklearn.ensemble.RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

**1) 앙상블(숲) 구조**
| 파라미터           | 의미                                   | 튜닝 포인트                                                      |
| -------------- | ------------------------------------ | ----------------------------------------------------------- |
| `n_estimators` | 트리 개수    | 많을수록 성능/안정↑ (시간↑)     |
| `bootstrap`    | 각 트리를 만들 때 부트스트랩 샘플링 사용 여부     | 디폴트=True (샘플링), False (전체 데이터 사용)   |
| `max_samples`  | (bootstrap=True일 때) 각 트리에 뽑는 샘플 비율 | 범위 (0~1) 과적합/속도 조절에 도움   |
| `oob_score`    | (bootstrap=True 필요) Out-of-bag 대략적인 자체 검증  | 디폴트=False |

**2) 모델 복잡도(과적합 제어)**
| 파라미터                    | 의미         | 튜닝 포인트              |
| ----------------------- | ---------- | ------------------- |
| `max_depth`             | 트리 최대 깊이 RF에서도 과적합 영향 큼 | 주로 4 ~ 12 |
| `min_samples_split`     | 분할 최소 샘플 수 | 디폴트=2↑하면 트리 단순화          |
| `min_samples_leaf`      | 리프 최소 샘플 수 | 디폴트=1↑하면 과적합↓    |
| `max_leaf_nodes`        | 리프 노드 상한   | 선택 옵션, 주로 10~1000 트리 크기 제한            |
| `min_impurity_decrease` | 분할 최소 개선량  | 디폴트=0 주로 0~0.01↑하면 보수적 분할          |

**3) 무작위 피처 샘플링 / 기타**
| 파라미터           | 의미                        | 튜닝 포인트                                                                                                          |
| -------------- | ------------------------- | --------------------------------------------------------------------------------------------------------------- |
| `max_features` | 노드 분할 시 고려할 feature 비율/개수 | **RF 성능 핵심 영향**. 디폴트 `1`(=전체), 보통 `sqrt`, `log2`, 혹은 비율(예: 0.3~0.8)로 튜닝 |
| `random_state` | 랜덤 시드   | 랜덤 값 고정 |
| `n_jobs`     | 병렬 처리   | `-1`이면 가능한 CPU 모두 사용       |
| `verbose`    | 로그 출력   | 디버깅 시 사용 |


## XGBRegressor 하이퍼파라미터  
> 이전 트리의 예측 오차(잔차)를 보완하는 방식으로 트리를 순차적으로 학습시키는  
Gradient Boosting 기반 앙상블 모델이다.  
학습 과정에서 정규화(Regularization)를 포함하여 과적합을 효과적으로 제어한다.
- 부스팅 앙상블 기법, 이전 오차를 점차 보완하는 구조
- 연속형 수치데이터가 많은 회귀 문제에서 좋은 성능
- 하이퍼 파라미터가 많아 튜닝 난이도 높은편, 블랙박스 성향 있음

참고링크: [xgboost.readthedocs](https://xgboost.readthedocs.io/en/stable/parameter.html)

**1) 부스팅 구조/ 학습률**
| 파라미터                    | 의미                  | 튜닝 포인트                                                                              |
| ----------------------- | ------------------- | ----------------------------------------------------------------------------------- |
| `n_estimators` | 부스팅 수(트리 개수)    | 많을수록 표현력↑ (과적합/시간↑)   |
| `learning_rate` | 한 번에 업데이트하는 크기  | 기본값 : 0.3 / 범위 : [0,1] 주로 작게(0.01~0.2) 시작 |

**2) 트리 구조(과적합 제어)/ 샘플링**
| 파라미터                         | 의미                 | 튜닝 포인트                         |
| ---------------------------- | ------------------ | ------------------------------ |
| `max_depth`                  | 트리 깊이              | 디폴트=6 깊을수록 복잡/과적합↑                   |
| `min_child_weight`           | 리프가 되기 위한 최소 가중치 합 | 디폴트=1 ↑하면 보수적(과적합↓)                  |
| `gamma`  | 분할을 허용하는 최소 손실 감소  | 디폴트=1 ↑하면 분할 덜함(보수적)                 |
| `subsample`   | 각 트리에 사용할 샘플 비율      | 0.5~1.0 에서 튜닝(과적합↓) |
| `colsample_bytree` | 트리마다 사용할 feature 비율  | 0.5~1.0 주로 튜닝   |
| `n_jobs`  | 병렬 처리  | CPU 병렬 `-1` |
| `random_state`    | 랜덤 시드    | 랜덤값 고정  |

**3) L1, L2 정규화(규제)**
| 파라미터             | 의미              | 튜닝 포인트                   |
| ---------------- | --------------- | ------------------------ |
| `reg_alpha`      | L1 규제   | 주로 0.01 ~ 1 Lasso 규제 (feature 선택 성향↑) |
| `reg_lambda`     | L2 규제   | 주로 0.1 ~ 10 Ridge 규제 (안정적) |



# DT, RF, XGB 회귀모델 성능평가

# 실습 목표
DAY 2) 각 머신러닝 모델의 하이퍼파라미터 튜닝을 통해 성능을 최대한 올려보세요.  
DAY 3) [kaggle](https://www.kaggle.com/t/2e8aa09a9f8740dfb3de0472326bf152)에 제출하여 리더보드 상위권을 노려보세요.

In [ ]:
# DT, RF, XGB 회귀모델 성능평가 - 실습용

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 데이터 EDA 혹은 하이퍼파라미터 튜닝으로 성능 향상 (random_state 팀 고유번호로 설정)

# 주요 튜닝 인자: max_depth, min_samples_split, min_samples_leaf, max_features
dt = DecisionTreeRegressor(random_state=42)
# 주요 튜닝 인자: max_depth, min_sapmles_split, min_samples_leaf, max_features, max_samples
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
# 주요 튜닝 인자: learning_rate, max_depth, min_child_weight, subsample, colsample_bytree, reg_lambda
xgb = XGBRegressor(n_estimators=100, n_jobs=-1, random_state=42)

In [ ]:
# EDA 데이터셋 학습 결과 - 실습용

# EDA 데이터셋 학습 및 성능평가

# Decision Tree
dt.fit(X_tr_EDA, y_train)
dt_pred = dt.predict(X_va_EDA)

# Random Forest
rf.fit(X_tr_EDA, y_train)
rf_pred = rf.predict(X_va_EDA)

# XGBoost
xgb.fit(X_tr_EDA, y_train)
xgb_pred = xgb.predict(X_va_EDA)

def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R2: {r2:.3f}")
    print("-"*30)

evaluate(y_val, dt_pred, "Decision Tree")
evaluate(y_val, rf_pred, "Random Forest")
evaluate(y_val, xgb_pred, "XGBoost")

# 실습 평가물 제출  

[kaggle 초대 링크](https://www.kaggle.com/t/2e8aa09a9f8740dfb3de0472326bf152)에 접속하여 우측상단 Join Competition 클릭  
참여 후, 다시 우측 상단, Submit Prediction을 클릭하여 파일을 제출하면 된다.  
Leaderboard에서 제출한 test 예측값 성능을 확인할 수 있으며, 하루 5회 제출 가능

In [ ]:
# kaggle 제출용 submission.csv파일 저장
submission = pd.read_csv('https://raw.githubusercontent.com/rngus4656/datasets/refs/heads/main/S2/sample_submission.csv')

# 최종 모델 예측값 저장 후 제출용 csv파일 생성
xgb_y_pred = xgb.predict(X_te_EDA)
submission['Rented Bike Count'] = xgb_y_pred
submission.to_csv('/content/xgb_submission.csv', index=False)

In [ ]:
# (선택) X_tr_EDA, X_va_EDA, X_te_EDA, y_train, y_val csv파일로 저장
X_tr_EDA.to_csv('/content/X_tr_EDA.csv', index=False)
X_va_EDA.to_csv('/content/X_va_EDA.csv', index=False)
X_te_EDA.to_csv('/content/X_te_EDA.csv', index=False)
y_train.to_csv('/content/y_train.csv', index=False)
y_val.to_csv('/content/y_val.csv', index=False)

#(심화 +) PCA 주성분 분석과 K-Means 클러스터링

## PCA 주성분 분석  
> PCA (Principal Component Analysis)  
데이터의 분산을 최대한 보존하는 방향으로 새로운 축(주성분, PC)을 생성하여 차원을 축소하는 기법이다.  
여러 변수들을 선형 결합하여 새로운 변수(주성분)로 변환한다.  
주로 고차원 데이터에서 정보 손실을 최소화하면서 차원을 줄이기 위해 사용된다

In [ ]:
# Screeplot

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(X_tr_EDA)

plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
         pca.explained_variance_ratio_,
         marker='o')
plt.title('Scree Plot')
plt.show()


In [ ]:
# PC 개수 설정
n_PC = 4

X_train_pca = X_tr_EDA.copy()
X_val_pca = X_va_EDA.copy()
X_test_pca = X_te_EDA.copy()

pca = PCA(n_components=n_PC)
train_pca = pca.fit_transform(X_tr_EDA)
val_pca = pca.transform(X_va_EDA)
test_pca = pca.transform(X_te_EDA)

pca_cols = [f'PC{i+1}' for i in range(n_PC)]
X_train_pca[pca_cols] = pd.DataFrame(train_pca, columns=pca_cols, index=X_tr_EDA.index)
X_val_pca[pca_cols] = pd.DataFrame(val_pca, columns=pca_cols, index=X_va_EDA.index)
X_test_pca[pca_cols] = pd.DataFrame(test_pca, columns=pca_cols, index=X_te_EDA.index)
X_train_pca.head()

## 비지도 학습 K-Means 클러스터링
> K-means Clustering  
데이터를 K개의 군집(Cluster)으로 나누는 비지도 학습 알고리즘이다.  
각 데이터는 가장 가까운 중심점(Centroid)을 기준으로 군집에 할당되며,  
군집 내 분산을 최소화하는 방향으로 중심점을 반복적으로 갱신한다.  
주로 데이터의 패턴 탐색, 그룹 분류, 세분화(Segmentation)에 사용된다.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

# PCA를 통한 K-means Clustering 군집 시각화

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tr_EDA)

kmeans4 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels4 = kmeans4.fit_predict(X_pca)

plt.figure(figsize=(6,6))
for cid in range(4):
    plt.scatter(
        X_pca[labels4 == cid, 0],
        X_pca[labels4 == cid, 1],
        label=f"Cluster {cid}"
    )

centers = kmeans4.cluster_centers_
plt.scatter(centers[:,0], centers[:,1], marker="X", s=200, label="Centroid")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clustering (PCA 2D, k=4)")
plt.legend()
plt.show()

In [ ]:
# 비지도 학습

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_tr_EDA)

X_train_cl = X_tr_EDA.copy()
X_train_cl['cluster'] = clusters
X_train_cl.head()

In [ ]:
X_train.head()

In [ ]:
X_train_cl['y_train'] = y_train

X_train_cl.groupby('cluster')['y_train'].agg(mean='mean', median='median', std='std').round(2)
